In [1]:
import pandas as pd
import numpy as np

# Load the original dataset
df_original = pd.read_excel(
    r"C:\Users\JAY\Desktop\92 Day Data Analysis Challenge\October - Advanced Excel Analysis\Week-04 Online Retail\Source data\Online Retail.xlsx"
)

# Create an independent working copy
df_prepared = df_original.copy()

print("Original Dataset Shape:", df_original.shape)
print("Working Dataset Shape:", df_prepared.shape)

print("\nWorking Dataset Preview:")
display(df_prepared.head())

Original Dataset Shape: (541909, 8)
Working Dataset Shape: (541909, 8)

Working Dataset Preview:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# Create a transaction value column in the working copy
df_prepared["TransactionValue"] = (
    df_prepared["Quantity"] * df_prepared["UnitPrice"]
)

# Preview the result
display(
    df_prepared[
        ["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "TransactionValue"]
    ].head(10)
)

print("Updated Working Dataset Shape:", df_prepared.shape)

,InvoiceNo,StockCode,Quantity,UnitPrice,TransactionValue
0,536365,85123A,6,2.55,15.30
1,536365,71053,6,3.39,20.34
2,536365,84406B,8,2.75,22.00
3,536365,84029G,6,3.39,20.34
4,536365,84029E,6,3.39,20.34
5,536365,22752,2,7.65,15.30
6,536365,21730,6,4.25,25.50
7,536366,22633,6,1.85,11.10
8,536366,22632,6,1.85,11.10
9,536367,84879,32,1.69,54.08


Updated Working Dataset Shape: (541909, 9)


In [3]:
# Identify StockCodes with exactly one distinct available description
description_counts = (
    df_original.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .nunique()
)

unique_description_codes = description_counts[
    description_counts == 1
].index

# Build the StockCode-to-Description mapping
description_mapping = (
    df_original[
        df_original["StockCode"].isin(unique_description_codes)
        & df_original["Description"].notna()
    ]
    .drop_duplicates(subset="StockCode")
    .set_index("StockCode")["Description"]
)

# Identify missing-description records with a candidate match
missing_description_mask = df_prepared["Description"].isna()

candidate_mask = (
    missing_description_mask
    & df_prepared["StockCode"].isin(description_mapping.index)
)

print("Missing descriptions:", missing_description_mask.sum())
print("Candidate records for recovery:", candidate_mask.sum())
print("Unique candidate StockCodes:", 
      df_prepared.loc[candidate_mask, "StockCode"].nunique())

# Preview candidate matches without changing the data
display(
    df_prepared.loc[
        candidate_mask,
        ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
    ].assign(
        CandidateDescription=lambda x: x["StockCode"].map(description_mapping)
    ).head(20)
)

Missing descriptions: 1454
Candidate records for recovery: 1033
Unique candidate StockCodes: 674


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CandidateDescription
1971,536546,22145,NaN,1,0.0,CHRISTMAS CRAFT HEART STOCKING
1972,536547,37509,NaN,1,0.0,NEW ENGLAND MUG W GIFT BOX
2025,536553,37461,NaN,3,0.0,FUNKY MONKEY MUG
2406,536589,21777,NaN,-10,0.0,RECIPE BOX WITH METAL HEART
4347,536764,84952C,NaN,-38,0.0,MIRROR LOVE BIRD T-LIGHT HOLDER
4348,536765,84952C,NaN,19,0.0,MIRROR LOVE BIRD T-LIGHT HOLDER
7188,536996,22712,NaN,-20,0.0,CARD DOLLY GIRL
7189,536997,22028,NaN,-20,0.0,PENNY FARTHING BIRTHDAY CARD
7190,536998,85067,NaN,-6,0.0,CREAM SWEETHEART WALL CABINET
7192,537000,21414,NaN,-22,0.0,SCALLOP SHELL SOAP DISH


In [4]:
# Create a flag to identify descriptions recovered during preparation
df_prepared["DescriptionRecovered"] = False

# Fill only missing descriptions with a unique StockCode match
df_prepared.loc[candidate_mask, "Description"] = (
    df_prepared.loc[candidate_mask, "StockCode"].map(description_mapping)
)

# Mark the records whose descriptions were recovered
df_prepared.loc[candidate_mask, "DescriptionRecovered"] = True

# Validate the result
print("Descriptions recovered:", df_prepared["DescriptionRecovered"].sum())
print("Descriptions still missing:", df_prepared["Description"].isna().sum())
print("Total rows:", len(df_prepared))

# Preview recovered records
display(
    df_prepared.loc[
        df_prepared["DescriptionRecovered"],
        ["InvoiceNo", "StockCode", "Description", "DescriptionRecovered"]
    ].head(10)
)

Descriptions recovered: 1033
Descriptions still missing: 421
Total rows: 541909


,InvoiceNo,StockCode,Description,DescriptionRecovered
1971,536546,22145,CHRISTMAS CRAFT HEART STOCKING,True
1972,536547,37509,NEW ENGLAND MUG W GIFT BOX,True
2025,536553,37461,FUNKY MONKEY MUG,True
2406,536589,21777,RECIPE BOX WITH METAL HEART,True
4347,536764,84952C,MIRROR LOVE BIRD T-LIGHT HOLDER,True
4348,536765,84952C,MIRROR LOVE BIRD T-LIGHT HOLDER,True
7188,536996,22712,CARD DOLLY GIRL,True
7189,536997,22028,PENNY FARTHING BIRTHDAY CARD,True
7190,536998,85067,CREAM SWEETHEART WALL CABINET,True
7192,537000,21414,SCALLOP SHELL SOAP DISH,True


In [5]:
# 1. Identify all records belonging to exact duplicate groups
df_prepared["IsExactDuplicate"] = df_original.duplicated(keep=False)

# 2. Identify additional occurrences of exact duplicates
df_prepared["IsAdditionalDuplicate"] = df_original.duplicated(keep="first")

# 3. Identify cancellation records
df_prepared["IsCancellation"] = (
    df_prepared["InvoiceNo"].astype(str).str.startswith("C")
)

# 4. Identify negative quantity and zero-price adjustments
df_prepared["IsZeroPriceNegativeAdjustment"] = (
    (df_prepared["Quantity"] < 0) &
    (df_prepared["UnitPrice"] == 0)
)

# 5. Identify bad-debt adjustment records
df_prepared["IsBadDebtAdjustment"] = (
    df_prepared["Description"].eq("Adjust bad debt")
)

# 6. Identify negative and zero unit prices
df_prepared["HasNegativeUnitPrice"] = df_prepared["UnitPrice"] < 0
df_prepared["HasZeroUnitPrice"] = df_prepared["UnitPrice"] == 0

# Display flag counts
flag_columns = [
    "IsExactDuplicate",
    "IsAdditionalDuplicate",
    "IsCancellation",
    "IsZeroPriceNegativeAdjustment",
    "IsBadDebtAdjustment",
    "HasNegativeUnitPrice",
    "HasZeroUnitPrice"
]

for column in flag_columns:
    print(f"{column}: {df_prepared[column].sum():,}")

IsExactDuplicate: 10,147
IsAdditionalDuplicate: 5,268
IsCancellation: 9,288
IsZeroPriceNegativeAdjustment: 1,336
IsBadDebtAdjustment: 3
HasNegativeUnitPrice: 2
HasZeroUnitPrice: 2,515


In [6]:
# Define the population used in the original outlier investigation
positive_sales_mask = (
    (df_prepared["Quantity"] > 0) &
    (df_prepared["UnitPrice"] > 0)
)

# Previously established IQR upper thresholds
quantity_threshold = 23.5
unitprice_threshold = 8.45
transactionvalue_threshold = 38.625

# Create outlier flags
df_prepared["IsQuantityOutlier"] = (
    positive_sales_mask &
    (df_prepared["Quantity"] > quantity_threshold)
)

df_prepared["IsUnitPriceOutlier"] = (
    positive_sales_mask &
    (df_prepared["UnitPrice"] > unitprice_threshold)
)

df_prepared["IsTransactionValueOutlier"] = (
    positive_sales_mask &
    (df_prepared["TransactionValue"] > transactionvalue_threshold)
)

# Display results
print("Quantity outliers:", df_prepared["IsQuantityOutlier"].sum())
print("Unit price outliers:", df_prepared["IsUnitPriceOutlier"].sum())
print("Transaction value outliers:", df_prepared["IsTransactionValueOutlier"].sum())

Quantity outliers: 56363
Unit price outliers: 37999
Transaction value outliers: 42651


In [7]:
# Final preparation validation

print("PREPARED DATASET VALIDATION")
print("-" * 40)

print("Total rows:", len(df_prepared))
print("Total columns:", len(df_prepared.columns))

print("\nOriginal columns preserved:")
original_columns = df_original.columns.tolist()
print(all(col in df_prepared.columns for col in original_columns))

print("\nMissing InvoiceNo:", df_prepared["InvoiceNo"].isna().sum())
print("Missing StockCode:", df_prepared["StockCode"].isna().sum())

print("\nDescriptions recovered:",
      df_prepared["DescriptionRecovered"].sum())

print("Descriptions still missing:",
      df_prepared["Description"].isna().sum())

print("\nTransactionValue calculation valid:",
      (df_prepared["TransactionValue"] ==
       df_prepared["Quantity"] * df_prepared["UnitPrice"]).all())

print("\nOriginal dataset unchanged:",
      df_original.shape == (541909, 8))

PREPARED DATASET VALIDATION
----------------------------------------
Total rows: 541909
Total columns: 20

Original columns preserved:
True

Missing InvoiceNo: 0
Missing StockCode: 0

Descriptions recovered: 1033
Descriptions still missing: 421

TransactionValue calculation valid: True

Original dataset unchanged: True


In [8]:
from pathlib import Path

# Define the output folder
output_folder = Path(
    r"C:\Users\JAY\Desktop\92 Day Data Analysis Challenge\October - Advanced Excel Analysis\Week-04 Online Retail\Deliverables"
)

# Create the folder if it does not already exist
output_folder.mkdir(parents=True, exist_ok=True)

# Define the output file
output_file = output_folder / "Online_Retail_Prepared.csv"

# Save the prepared dataset
df_prepared.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset saved successfully!")
print("File location:", output_file)
print("Rows saved:", len(df_prepared))
print("Columns saved:", len(df_prepared.columns))

Dataset saved successfully!
File location: C:\Users\JAY\Desktop\92 Day Data Analysis Challenge\October - Advanced Excel Analysis\Week-04 Online Retail\Deliverables\Online_Retail_Prepared.csv
Rows saved: 541909
Columns saved: 20


In [9]:
# Load the saved CSV into a separate DataFrame
df_saved = pd.read_csv(output_file)

# Verify the saved file
print("SAVED FILE VALIDATION")
print("-" * 40)

print("Rows:", len(df_saved))
print("Columns:", len(df_saved.columns))

print("\nTransactionValue calculation valid:",
      np.allclose(
          df_saved["TransactionValue"],
          df_saved["Quantity"] * df_saved["UnitPrice"],
          equal_nan=True
      ))

print("\nRecovered descriptions:",
      df_saved["DescriptionRecovered"].sum())

print("Exact duplicate occurrences:",
      df_saved["IsAdditionalDuplicate"].sum())

print("\nSaved file preview:")
display(df_saved.head())

SAVED FILE VALIDATION
----------------------------------------
Rows: 541909
Columns: 20

TransactionValue calculation valid: True

Recovered descriptions: 1033
Exact duplicate occurrences: 5268

Saved file preview:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionValue,DescriptionRecovered,IsExactDuplicate,IsAdditionalDuplicate,IsCancellation,IsZeroPriceNegativeAdjustment,IsBadDebtAdjustment,HasNegativeUnitPrice,HasZeroUnitPrice,IsQuantityOutlier,IsUnitPriceOutlier,IsTransactionValueOutlier
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,False,False,False,False,False,False,False,False,False,False,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False,False,False,False,False,False,False,False,False,False,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,False,False,False,False,False,False,False,False,False,False,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False,False,False,False,False,False,False,False,False,False,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False,False,False,False,False,False,False,False,False,False,False
